In [9]:
# open csv file and read it
import csv
import os

cq_with_classes = []
csv_path = os.path.join(os.getcwd(), "datasets", "data", "cqs_and_classes.csv")
with open(csv_path, "r") as file:
    reader = csv.reader(file)
    for row in reader:
        cq_with_classes.append({"cq": row[0], "classes": row[1].split("\n")})

In [10]:
cq_with_classes

[{'cq': 'What are all the enslaved people in County X in the century X?',
  'classes': ['Agent',
   'AgentRecord',
   'PersonStatusRecord',
   'Place',
   'PlaceCV',
   'TemporalExtent',
   'NameRecord']},
 {'cq': 'What are all the enslaved individuals categorized by gender, ethnicity, and location (e.g., on plantation X or of XXXX ethnicity)?',
  'classes': ['Agent',
   'AgentRecord',
   'PersonStatusRecord',
   'SexRecord',
   'RaceRecord',
   'Place',
   'PlaceCV']},
 {'cq': 'What are all the available population counts of enslaved people by place and year?',
  'classes': ['AgentRecord',
   'PersonStatusRecord',
   'Place',
   'PlaceCV',
   'TemporalExtent',
   'Timespan']},
 {'cq': 'What are all the documented literacy statuses of enslaved individuals?',
  'classes': ['Agent', 'AgentRecord', 'PersonStatusRecord', 'Description']},
 {'cq': 'What are all the geographical places associated with enslaved people and their relationships to those places?',
  'classes': ['Agent',
   'AgentR

In [11]:
def calculate_accuracy_falsehits_jaccard(true_set, predicted_set):
    true_positive = true_set & predicted_set
    # Classes that WERE predicted correctly

    false_negative = true_set - predicted_set
    # Classes that SHOULD have been predicted but weren't (False Negatives)

    false_positive = predicted_set - true_set
    # Classes that were predicted but shouldn't have been (False Positives)

    # Hits (correct predictions)
    hits = true_set & predicted_set

    # False hits (incorrect predictions)
    false_hits = predicted_set - true_set

    # Overlap accuracy
    accuracy = len(hits) / len(predicted_set)

    jaccard_similarity = (
        len(true_positive) / (len(true_set) + len(predicted_set) - len(true_positive))
        if (len(true_set) + len(predicted_set) - len(true_positive)) > 0
        else 0
    )
    return accuracy, false_hits, jaccard_similarity

In [ ]:
import json


def get_evaluation_results(file_path):
    file_name = file_path.split("/")[-1]
    path = file_path.replace(file_name, "")
    evaluated_file_name = file_name.replace(".json", ".csv")
    evaluated_file_path = os.path.join(path, evaluated_file_name)
    with open(evaluated_file_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["CQ", "True Classes", "Predicted Classes", "Model", "Accuracy", "False hits", "Jaccard Similarity"])

    # read from json file

    with open(
        file_path, "r"
    ) as json_file:
        output = json.load(json_file)
        for item in cq_with_classes:
            cq = item["cq"]
            true_classes = set(item["classes"])
            for each in output.get("summary", []):
                if each["cq"] == cq:
                    for each_model in each["per_model"]:
                        model = each_model["model"]
                        predicted_classes = set(
                            [a["class"] for a in each_model.get("classes", [])]
                        )
                        accuracy, false_hits, jaccard_similarity = calculate_accuracy_falsehits_jaccard(
                            true_classes, predicted_classes
                        )
                        with open(
                            evaluated_file_path, "a", newline=""
                        ) as csvfile:
                            writer = csv.writer(csvfile)
                            writer.writerow([cq, true_classes, predicted_classes, model, accuracy, len(false_hits), jaccard_similarity])

In [13]:
get_evaluation_results("datasets/output/output_dynamic_raw_embedding_results.json")